# Step 1 — Waste Classifier Training (Garbage Classification, 12 classes)

**Run this notebook in Google Colab** (Runtime → Change runtime type → GPU).

This notebook:
1. Downloads the `mostafaabla/garbage-classification` dataset from Kaggle (12 classes).
2. Builds a transfer-learning model on **MobileNetV2**.
3. Trains, evaluates, and saves the model.
4. Downloads the trained model file so you can bring it back and plug it into the app.

You'll need a free Kaggle account and an API token (`kaggle.json`) — instructions are in Cell 2.


In [ ]:
!pip install -q kagglehub tensorflow


## Get the dataset

Easiest path: `kagglehub` handles the Kaggle download without you manually uploading a `kaggle.json` file, as long as you're logged into a Kaggle account in this browser session (it will prompt you to authenticate).

If prompted for a Kaggle API token instead:
1. Go to kaggle.com → your profile → Account → “Create New API Token” — this downloads `kaggle.json`.
2. Upload that file when Colab asks for it.


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("mostafaabla/garbage-classification")
print("Dataset downloaded to:", dataset_path)

import os
print(os.listdir(dataset_path))


In [ ]:
import os

# The dataset extracts to a folder that contains one subfolder per class.
# Find that folder automatically (handles cases where it's nested one level deep).
def find_class_dir(root):
    for cur, dirs, files in os.walk(root):
        # A class dir is one whose subfolders all contain images and there are several of them
        if len(dirs) >= 10:
            return cur
    return root

DATA_DIR = find_class_dir(dataset_path)
print("Using data directory:", DATA_DIR)
print("Classes found:", sorted(os.listdir(DATA_DIR)))


## Build train/validation datasets

12 classes, resized to 224x224 (MobileNetV2's expected input size), 80/20 train-val split.


In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Number of classes:", num_classes)

# Save class order now -- we'll need this exact list & order later in the app
import json
with open("class_names.json", "w") as f:
    json.dump(class_names, f)


In [ ]:
# Performance: cache to DISK (not RAM), then prefetch.
# NOTE: no extra .shuffle() here -- image_dataset_from_directory already shuffles
# file order internally by default. Adding .shuffle() on an already-batched
# dataset with a large buffer forces TF to hold the whole dataset in memory,
# which is what crashed the Colab session last time.
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache("/content/train_cache").prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache("/content/val_cache").prefetch(buffer_size=AUTOTUNE)

# Light data augmentation to reduce overfitting (household photos vary a lot in angle/lighting)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


## Build the model — MobileNetV2 transfer learning

Phase 1: freeze the pretrained base, train only the new classification head.
Phase 2 (optional fine-tuning): unfreeze the top layers of the base and train a few more epochs at a low learning rate.


In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # freeze for phase 1

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True
)

EPOCHS = 12

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop],
)


## (Optional) Phase 2: fine-tuning

Unfreeze the top portion of MobileNetV2 and continue training at a much lower learning rate.
This usually adds a few more accuracy points. Skip this cell if Phase 1 accuracy is already good enough (85%+) and you're short on time.


In [ ]:
base_model.trainable = True

# Freeze everything except the last ~30 layers
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_epochs = 6
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=fine_tune_epochs,
    callbacks=[early_stop],
)


## Evaluate & plot

In [ ]:
import matplotlib.pyplot as plt

val_loss, val_acc = model.evaluate(val_ds)
print(f"Final validation accuracy: {val_acc:.3f}")

plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.legend()
plt.title("Accuracy over epochs")
plt.show()


## Save & download the model

Saves in the modern Keras format (`.keras`). Bring this file (plus `class_names.json`) back to continue with the app.


In [ ]:
model.save("waste_classifier.keras")
print("Saved model and class_names.json")

from google.colab import files
files.download("waste_classifier.keras")
files.download("class_names.json")
